[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance3_exercices.ipynb)

# Séance 2.3 — Agréger et croiser plusieurs tables

**Exercices** · durée : 2h — cinq techniques, chacune suivie de deux exercices

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- filtrer sur plusieurs conditions sans se noyer dans les parenthèses
- classer et extraire un top 5 en une commande
- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions avec un tableau croisé

## Pour aller plus loin

Les exercices de la séance sont dans votre **notebook de cours** : c'est là qu'on
travaille ensemble. Cette feuille-ci est **facultative**.

Elle reprend les mêmes techniques sur d'autres questions — à faire quand vous avez fini
avant les autres, ou tranquillement après la séance. Certains exercices ont des `____` à
remplir, d'autres une cellule vide où vous écrivez tout ; ceux qui se terminent par une
cellule de **vérification** vous disent immédiatement si votre réponse est bonne.

> 💡 Pas de vérification partout. Affichez systématiquement votre résultat et demandez-vous
> s'il est **plausible** : c'est le seul contrôle dont vous disposerez en entreprise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]
vc = ventes.merge(clients, on="client_id")     ## comme en seance
complet = vc.merge(produits, on="prod_id")     ## les trois fichiers
print(len(ventes), len(vc), len(complet))

### Exercice 1 — Le meilleur client

> **Votre mission :**
> - Calculer le chiffre d'affaires par client.
> - Mettre l'identifiant du meilleur dans `meilleur_client` et son CA dans `ca_meilleur` (arrondi à 2 décimales).

In [ ]:
ca_client = ventes.groupby("____")["ca"].____()

meilleur_client = ca_client.idxmax()
ca_meilleur = round(ca_client.____(), 2)

print(meilleur_client, ":", ca_meilleur, "euros")

In [ ]:
verifier("1a - meilleur client", meilleur_client == 14911,
         "groupby sur client_id puis sum() sur la colonne ca")
verifier("1b - son chiffre d'affaires", ca_meilleur == 143825.06,
         "idxmax() renvoie l'identifiant, max() renvoie le montant")

### Exercice 2 — Le chiffre d'affaires par pays

> **Votre mission :**
> - À partir de `vc`, calculer le CA par pays, trié du plus grand au plus petit → `ca_pays`.
> - Mettre le CA de la France dans `ca_france` (arrondi à 2 décimales).

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=____)
ca_france = round(ca_pays["____"], 2)

print(ca_pays.head(3).round(2))
print("France :", ca_france)

In [ ]:
verifier("2 - CA de la France", ca_france == 133984.8,
         "groupby('pays') puis sum() sur ca, et ca_pays['France']")

### Question 3 — Combien de pays font 80 % du chiffre d'affaires ?

> **Votre mission :**
> - Calculer la part de chaque pays dans le CA total, en %, triée du plus grand au plus petit.
> - Puis le **cumul** de ces parts, et enfin le nombre de pays nécessaires pour atteindre 80 %.
> - *Nouveau :* `serie.cumsum()` additionne au fur et à mesure.

### Question 4 — Le panier moyen, pays par segment

> **Votre mission :**
> - Attention au piège : un panier est une **commande**, pas une ligne. Il faut donc d'abord agréger par `cmd_id`.
> - Construire ensuite un tableau croisé pays × segment contenant le **panier moyen**.
> - Certaines cases sont vides. Est-ce une erreur ?

### Question 5 — La composition du panier, en %

> **Votre mission :**
> - Pour les quatre pays les plus présents, quelle **part** de leurs lignes chaque catégorie représente-t-elle ?
> - Des effectifs bruts ne se comparent pas entre un pays de 20 000 lignes et un pays de 2 000. Des pourcentages, si.
> - *Nouveau :* `pd.crosstab(a, b, normalize='index')` ramène chaque **ligne** à 100 %.

### Question 6 — La jointure qui ment

> **Votre mission :**
> - Ne garder que les produits de la catégorie `cuisine`, puis les joindre à `ventes` de deux façons : un `merge` normal, et un `merge(how='left', indicator=True)`.
> - Combien de lignes chaque version renvoie-t-elle ? Combien la première fait-elle disparaître **sans le dire** ?
> - *Nouveau :* `how='left'` garde toutes les lignes de gauche ; `indicator=True` ajoute une colonne `_merge` qui dit d'où vient chaque ligne.

### Question 7 — Sur quels marchés sommes-nous exposés ?

> **Votre mission :**
> - Le comité veut savoir **sur quels marchés l'entreprise est vraiment exposée**.
> - Produire un tableau par pays avec : le CA, le nombre de clients distincts, et le CA moyen par client.
> - Le trier par CA décroissant et ne garder que les cinq premiers.
> - Puis, en commentaire, la phrase que vous mettriez sous ce tableau.